# MoP + DivPO — Phase 2 & 3 (Kaggle)

**Prerequisite:** SFT adapters already at `DasonTio/mop-divpo-coauthor/sft/{persona}/`.

## Kaggle settings (before running)

| Setting | Value |
|---|---|
| Accelerator | **T4 x2** (Settings → Accelerator → GPU T4 x2) |
| Internet | On |
| Secret | `HF_TOKEN` = your HuggingFace write token |

## How to run in background (Save Version)

1. Click **Save Version** (top-right)
2. Choose **Save & Run All (Commit)**
3. Click **Save** — Kaggle runs the full notebook on its servers
4. Close browser / sleep laptop — job continues
5. Kaggle emails you when done. Check output at **Your Work → Notebooks**

> Each persona pushes to HF Hub immediately after completing (`--push`).
> If Kaggle hits the time limit mid-run, completed personas are already saved.

## Session plan (with GPU optimizations)

| Run | Cells | Est. time |
|---|---|---|
| Save Version 1 | Phase 2: data gen | **~40 min** (was 2–4h) |
| Save Version 2 | Phase 3: DivPO training | **~2–2.5h** (was ~4h) |

## What changed for GPU performance

| Optimization | Speedup |
|---|---|
| Batched generation (`--gen-batch-size 16`) — 500 serial calls → 32 batched | ~5x |
| Embedder on `cuda:1` — frees `cuda:0` entirely for LLM | avoids contention |
| Batch encode all texts per batch — 12,000 tiny encode calls → 32 batched | ~10x |
| DivPO training: ref model on `cuda:1`, batch 8 (was 4) | ~2x throughput |

---
## Cell 1 — Install dependencies

In **Save Version mode** no kernel restart is needed — fresh batch kernel hasn't imported anything yet.

In **interactive mode**: run this cell, restart kernel once, then continue from Cell 2.

In [ ]:
import os
INTERACTIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao

if INTERACTIVE:
    print("\n>>> INTERACTIVE MODE: restart kernel now, then continue from Cell 2. <<<")
else:
    print("Save Version mode — continuing.")

---
## Cell 2 — Setup

Credentials + repo + working directory. **Run this at the start of every session.**

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/kaggle/working/mop-divpo-llm-counter-argument"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DasonTio/mop-divpo-llm-counter-argument.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --quiet

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpu == 0:
    print("NO GPU — go to Notebook settings → Accelerator → T4 x2")
print(f"CWD : {os.getcwd()}")
print(f"HF  : {os.environ['HF_TOKEN'][:8]}...")
print("Ready.")

---
## Phase 2 — DivPO Dataset Generation

- LLM generation on `cuda:0` — batched (`--gen-batch-size 16`): 500 prompts → 32 batched calls
- Embedder scoring on `cuda:1` — one encode call per batch instead of per candidate
- Pushes to HF Hub after each persona

**Est. ~10 min per persona on T4 x2 (was 30–60 min).**

In [ ]:
# Download SFT JSONL prompt pool (skips if already present)
from huggingface_hub import hf_hub_download
import os

os.makedirs("data/processed/sft", exist_ok=True)
for persona in ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]:
    path = f"data/processed/sft/{persona}.jsonl"
    if os.path.exists(path):
        print(f"  {persona}.jsonl already present")
        continue
    hf_hub_download(
        repo_id="DasonTio/mop-divpo-sft-data",
        filename=f"{persona}.jsonl",
        repo_type="dataset",
        local_dir="data/processed/sft",
        token=os.environ["HF_TOKEN"],
    )
    print(f"  Downloaded {persona}.jsonl")

In [ ]:
# DivPO pairs — contrarian
# --gen-batch-size 16: generates 16 prompts × 4 candidates = 64 sequences per GPU call
!python scripts/prepare_divpo_datasets.py --persona contrarian --from-hub --candidate-count 4 --gen-batch-size 16 --push

In [ ]:
# DivPO pairs — systems_thinker
!python scripts/prepare_divpo_datasets.py --persona systems_thinker --from-hub --candidate-count 4 --gen-batch-size 16 --push

In [ ]:
# DivPO pairs — cross_domain_analogist
!python scripts/prepare_divpo_datasets.py --persona cross_domain_analogist --from-hub --candidate-count 4 --gen-batch-size 16 --push

In [ ]:
# DivPO pairs — minimalist
!python scripts/prepare_divpo_datasets.py --persona minimalist --from-hub --candidate-count 4 --gen-batch-size 16 --push

In [ ]:
# Verify all 4 persona files uploaded to HF Hub
from huggingface_hub import list_repo_files
import os

files = sorted(list_repo_files("DasonTio/mop-divpo-divpo-data", repo_type="dataset", token=os.environ["HF_TOKEN"]))
print("DivPO data on Hub:")
for f in files:
    print(" ", f)
expected = {"contrarian.jsonl", "systems_thinker.jsonl", "cross_domain_analogist.jsonl", "minimalist.jsonl"}
missing = expected - set(files)
print("\nMissing:", missing if missing else "none — Phase 2 complete")

---
## Phase 3 — DivPO Training

- Trainable model on `cuda:0`, frozen ref model on `cuda:1` — no VRAM contention
- Batch size 8 (was 4), grad accum 4 (was 8) — same effective batch of 32, 2x throughput
- Pushes to `DasonTio/mop-divpo-coauthor/divpo/{persona}/` after each persona

**Est. ~30–35 min per persona on T4 x2 (was ~60 min).**

> Starting a new Save Version for Phase 3? Cell 2 runs automatically first.

In [ ]:
# DivPO training — contrarian
# model on cuda:0, ref_model on cuda:1, batch_size=8
!python scripts/train_divpo.py --persona contrarian

In [ ]:
# DivPO training — systems_thinker
!python scripts/train_divpo.py --persona systems_thinker

In [ ]:
# DivPO training — cross_domain_analogist
!python scripts/train_divpo.py --persona cross_domain_analogist

In [ ]:
# DivPO training — minimalist
!python scripts/train_divpo.py --persona minimalist

---
## Verify — Load DivPO adapter and generate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")

# Change subfolder to test other personas: divpo/systems_thinker, divpo/minimalist, etc.
model = PeftModel.from_pretrained(base, "DasonTio/mop-divpo-coauthor", subfolder="divpo/contrarian")
model.eval()

prompt = "Generate a counter-argument to this claim:\n\nRemote work is strictly better for productivity."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.9, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))